# 02 Data Cleaning

## 1. Objective

The objective of this notebook is to clean and prepare the raw debt-collection datasets for exploratory analysis and predictive modelling. Cleaning decisions will be based on data quality, business context, and the intended prediction task. Significant transformations and exclusions will be documented to maintain reproducibility.

In [1]:
import pandas as pd
import numpy as np

## 2. Load Data

The raw debt-collection datasets are loaded into pandas DataFrames for further inspection and cleaning. The original files are kept unchanged so that all cleaning and transformation steps can be reproduced.

In [8]:
from pathlib import Path

print(Path.cwd())

/Users/jamie-leejoseph/Documents/debt-collection-prediction/notebooks


In [9]:
from pathlib import Path

print(Path("../data").exists())
print(list(Path("../data").iterdir()))

True
[PosixPath('../data/processed'), PosixPath('../data/raw')]


In [10]:
print(list(Path("../data/raw").iterdir()))

[PosixPath('../data/raw/PTP.xlsx'), PosixPath('../data/raw/Matters.xlsx'), PosixPath('../data/raw/Payments.xlsx'), PosixPath('../data/raw/UKZN_EnrichedData2.xlsx'), PosixPath('../data/raw/UKZN_EnrichedData1.xlsx'), PosixPath('../data/raw/CallHistory.xlsx'), PosixPath('../data/raw/EnrichedData_Combined.xlsx')]


In [11]:
matters = pd.read_excel(
    "../data/raw/Matters.xlsx",
    sheet_name="Data"
)

In [12]:
payments = pd.read_excel(
    "../data/raw/Payments.xlsx",
    sheet_name="Data"
)

In [13]:
ptp = pd.read_excel(
    "../data/raw/PTP.xlsx",
    sheet_name="Data"
)

In [14]:
call_history = pd.read_excel(
    "../data/raw/CallHistory.xlsx",
    sheet_name="Data"
)

In [16]:
enriched_combined = pd.read_excel(
    "../data/raw/EnrichedData_Combined.xlsx"
)

In [19]:
print(f"Matters:")
print(f"Rows: {matters.shape[0]:,}")
print(f"Columns: {matters.shape[1]:,}")

Matters:
Rows: 56,179
Columns: 12


In [20]:
print(f"Call History:")
print(f"R0ws: {call_history.shape[0]:,}")
print(f"Columns: {call_history.shape[1]:,}")

Call History:
R0ws: 430,197
Columns: 12


In [21]:
print(f"Payments:")
print(f"Rows: {payments.shape[0]:,}")
print(f"Columns: {payments.shape[1]:,}")

Payments:
Rows: 6,016
Columns: 8


In [22]:
print(f"PTPs:")
print(f"Rows: {ptp.shape[0]:,}")
print(f"Columns: {ptp.shape[1]:,}")


PTPs:
Rows: 9,402
Columns: 16


In [23]:
print(f"Enriched:")
print(f"Rows: {enriched_combined.shape[0]:,}")
print(f"Columnns: {enriched_combined.shape[1]:,}")

Enriched:
Rows: 55,086
Columnns: 517


## 3. Cleaning Preparation

The initial data understanding conducted in the previous notebook identified the structure, data types, missing-value patterns, duplicates, and potential data-quality issues within the datasets. This notebook builds on those findings and focuses on applying and documenting the required cleaning and preparation steps.

In [24]:
matters.info()

<class 'pandas.DataFrame'>
RangeIndex: 56179 entries, 0 to 56178
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   M_IDX             56179 non-null  int64         
 1   HandoverDate      56179 non-null  datetime64[us]
 2   CurrentStatusID   56179 non-null  int64         
 3   CurrentStatus     56179 non-null  str           
 4   PreviousStatusID  1152 non-null   float64       
 5   PreviousStatus    1152 non-null   str           
 6   FirstPaymentDate  3771 non-null   datetime64[us]
 7   ActivationPeriod  3771 non-null   float64       
 8   OpeningBalance    56179 non-null  float64       
 9   CurrentBalance    55240 non-null  float64       
 10  Industry          56179 non-null  str           
 11  DateCreated       56179 non-null  datetime64[us]
dtypes: datetime64[us](3), float64(4), int64(2), str(3)
memory usage: 5.1 MB


In [25]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 6016 entries, 0 to 6015
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   M_IDX                6016 non-null   int64         
 1   PaymentDate          6016 non-null   datetime64[us]
 2   AmountPaid           6016 non-null   float64       
 3   IsPayedAtClient      6016 non-null   int64         
 4   IsDebitOrderPayment  6016 non-null   int64         
 5   PaymentMethodID      6016 non-null   int64         
 6   PaymentDescription   6016 non-null   str           
 7   AgentID              6016 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(5), str(1)
memory usage: 376.1 KB


In [26]:
call_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 430197 entries, 0 to 430196
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   M_IDX               430197 non-null  int64         
 1   AgentID             430197 non-null  int64         
 2   HistoryID           430197 non-null  int64         
 3   CallDate            430197 non-null  datetime64[us]
 4   MinutesOfCall       204066 non-null  float64       
 5   HasRPC              82508 non-null   float64       
 6   CallTypeID          350124 non-null  float64       
 7   CallType            430197 non-null  str           
 8   PTPCreateIndicator  8209 non-null    float64       
 9   TimeCallStart       209016 non-null  str           
 10  TimeCallConfirmRPC  17416 non-null   str           
 11  TimeCallEnded       204066 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(3), str(4)
memory usage: 39.4 MB


In [27]:
ptp.info()

<class 'pandas.DataFrame'>
RangeIndex: 9402 entries, 0 to 9401
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   M_IDX                    9402 non-null   int64         
 1   PTPCreateDate            9402 non-null   datetime64[us]
 2   PaymentMethodID          9402 non-null   int64         
 3   PaymentDescription       9402 non-null   str           
 4   PaymentFrequencyID       9402 non-null   int64         
 5   PaymentFrequency         7567 non-null   str           
 6   FirstPaymentAmount       9241 non-null   float64       
 7   FirstPaymentDate         3228 non-null   datetime64[us]
 8   MonthlyPaymentAmount     9402 non-null   float64       
 9   MonthlyPaymentStartDate  7568 non-null   datetime64[us]
 10  DebitAmount              11 non-null     float64       
 11  CreditAmount             9402 non-null   float64       
 12  ProjectionPaymentDate    9402 non-null   date

In [28]:
enriched_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 55086 entries, 0 to 55085
Columns: 517 entries, M_IDX to Cell1Score
dtypes: datetime64[us](1), float64(356), int64(10), object(41), str(109)
memory usage: 217.3+ MB


## 4. Matters Cleaning

### 4.1 Duplicate Matters
Duplicate records can affect the reliability of the modelling dataset. The M_IDX field is used to identify whether the same matter appears more than once.

In [31]:
matters["M_IDX"].nunique()


56179

In [32]:
matters["M_IDX"].is_unique

True

**Finding:** Each matter has a unique `M_IDX`. There are 56,179 unique matter IDs across 56,179 rows, so no duplicate matters were identified and no rows were removed at this stage.

In [33]:
matters[matters["CurrentBalance"].isna()]["CurrentStatus"].value_counts()

CurrentStatus
Closed    939
Name: count, dtype: int64

In [34]:
matters[matters["CurrentBalance"].isna()]["OpeningBalance"].describe()

count       939.000000
mean      14687.616539
std       17730.374895
min         144.680000
25%        3715.200000
50%        9142.220000
75%       18844.820000
max      181597.150000
Name: OpeningBalance, dtype: float64

In [35]:
matters.groupby("CurrentStatus")[["OpeningBalance", "CurrentBalance"]].agg(["count", "mean"])

OpeningBalance               CurrentBalance              
                             count          mean          count          mean
CurrentStatus                                                                
Attempting PTP               26474   8370.901968          26474   8486.157391
Authentication                 271  10786.158081            271  10860.610996
Broken PTP                    2817   7329.035279           2817   7384.717636
Closed                       15882  11734.165002          14943  11469.760656
Dispute                          1   3435.280000              1   3616.330000
Follow-up PTP                   70   5319.022143             70   3741.016143
In Progress                    232   9604.242198            232   9601.309741
New Instruction               6341  11512.631798           6341  11536.247649
On Hold                        160   3122.188000            160   2654.922125
Payment Arrangement           3633   9589.225092           3633   9491.445866
Re-opened                      292   1862.664041            292   1808.531952
Request for Closure              6   4274.270000              6   4213.498333

### 4.3 Opening Balance and Current Balance
The relationship between OpeningBalance and CurrentBalance is investigated to identify matters where no change in balance has occurred. These records will be assessed in the context of the business rules and modelling population before any exclusions are made.

In [36]:
equal_balance = matters["OpeningBalance"] == matters["CurrentBalance"]

equal_balance.sum()

np.int64(15604)

In [37]:
matters.loc[equal_balance, "CurrentStatus"].value_counts()

CurrentStatus
Closed                 9515
Attempting PTP         3623
New Instruction         949
Broken PTP              695
Payment Arrangement     428
Re-opened               267
On Hold                  59
In Progress              54
Authentication           12
Follow-up PTP             2
Name: count, dtype: int64

In [38]:
closed_equal_balance = (
    (matters["CurrentStatus"] == "Closed") &
    equal_balance
)

closed_equal_balance.sum()

np.int64(9515)

In [39]:
matters.loc[
    closed_equal_balance,
    "FirstPaymentDate"
].isna().sum()

np.int64(9381)

In [40]:
matters.loc[
    closed_equal_balance & matters["FirstPaymentDate"].notna(),
    ["M_IDX", "CurrentStatus", "FirstPaymentDate", "OpeningBalance", "CurrentBalance"]
].head(10)

,M_IDX,CurrentStatus,FirstPaymentDate,OpeningBalance,CurrentBalance
13,865780,Closed,2025-02-26,210.90,210.90
19,865866,Closed,2025-04-30,179.71,179.71
24,865884,Closed,2025-03-14,190.85,190.85
27,865900,Closed,2025-03-31,280.00,280.00
28,865913,Closed,2025-03-20,1004.76,1004.76
31,865918,Closed,2025-03-07,171.86,171.86
32,865924,Closed,2025-02-26,280.00,280.00
40,865978,Closed,2025-03-14,632.66,632.66
48,866005,Closed,2025-02-28,1484.10,1484.10
54,866033,Closed,2025-05-19,656.03,656.03


**Finding:** 9,381 matters were identified as retracted accounts based on the combination of `CurrentStatus = "Closed"`, `OpeningBalance = CurrentBalance`, and a missing `FirstPaymentDate`. The remaining 134 Closed matters with unchanged balances had a recorded `FirstPaymentDate` and were therefore not classified as retracted accounts.


In [41]:
retracted_accounts = (
    (matters["CurrentStatus"] == "Closed") &
    equal_balance &
    matters["FirstPaymentDate"].isna()
)
retracted_accounts.sum()

np.int64(9381)

In [42]:
matters = matters.loc[~retracted_accounts].copy()

In [43]:
matters.shape

(46798, 12)

In [44]:
retracted_accounts.sum()

np.int64(9381)

In [45]:
(
    (matters["CurrentStatus"] == "Closed") &
    (matters["OpeningBalance"] == matters["CurrentBalance"]) &
    (matters["FirstPaymentDate"].isna())
).sum()

np.int64(0)

In [46]:
matters["CurrentBalance"].isna().sum()

np.int64(939)

In [47]:
matters.loc[
    matters["CurrentBalance"].isna(),
    "FirstPaymentDate"
].isna().value_counts()

FirstPaymentDate
True     937
False      2
Name: count, dtype: int64

In [48]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].isna(),
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].head(10)

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
2623,1196512,Closed,6908.65,NaN,NaT
2624,1196513,Closed,94001.07,NaN,NaT
2625,1196514,Closed,8326.18,NaN,NaT
2626,1196515,Closed,21390.80,NaN,NaT
2627,1196516,Closed,10254.58,NaN,NaT
2628,1196517,Closed,4303.33,NaN,NaT
2629,1196518,Closed,5165.08,NaN,NaT
2630,1196519,Closed,17559.64,NaN,NaT
2631,1196520,Closed,9912.78,NaN,NaT
2632,1196521,Closed,5874.04,NaN,NaT


In [49]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].notna(),
    ["M_IDX", "CurrentStatus", "HandoverDate", "FirstPaymentDate",
     "OpeningBalance", "CurrentBalance", "DateCreated"]
]

,M_IDX,CurrentStatus,HandoverDate,FirstPaymentDate,OpeningBalance,CurrentBalance,DateCreated
5719,1169872,Closed,2025-02-12,2025-02-17,3013.90,NaN,2025-05-23
35130,1490892,Closed,2025-03-14,2025-03-26,1965.88,NaN,2025-05-23


In [50]:
payments.columns

Index(['M_IDX', 'PaymentDate', 'AmountPaid', 'IsPayedAtClient',
       'IsDebitOrderPayment', 'PaymentMethodID', 'PaymentDescription',
       'AgentID'],
      dtype='str')

In [51]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(payments["M_IDX"]).sum()

np.int64(0)

In [52]:
ptp.columns

Index(['M_IDX', 'PTPCreateDate', 'PaymentMethodID', 'PaymentDescription',
       'PaymentFrequencyID', 'PaymentFrequency', 'FirstPaymentAmount',
       'FirstPaymentDate', 'MonthlyPaymentAmount', 'MonthlyPaymentStartDate',
       'DebitAmount', 'CreditAmount', 'ProjectionPaymentDate',
       'ProjectionDescriptionID', 'ProjectionDescription', 'HistoryID'],
      dtype='str')

In [53]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(ptp["M_IDX"]).sum()

np.int64(0)

In [54]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    ["HandoverDate", "DateCreated"]
].value_counts()

HandoverDate  DateCreated
2025-03-04    2025-05-23     914
2025-02-12    2025-05-23       7
2025-02-10    2025-05-23       5
2025-02-14    2025-05-23       4
2025-02-13    2025-05-23       2
2025-03-14    2025-05-23       2
2025-02-05    2025-05-23       1
2025-02-04    2025-05-23       1
2025-03-10    2025-05-23       1
Name: count, dtype: int64

In [55]:
two_missing_balance = (
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].notna()
)

matters.loc[two_missing_balance, "M_IDX"].isin(payments["M_IDX"]).sum()

np.int64(2)

In [56]:
two_midx = matters.loc[two_missing_balance, "M_IDX"]

payments[payments["M_IDX"].isin(two_midx)]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
1654,1169872,2025-02-18,3423.71,0,0,1,Direct Deposit,77502
3763,1490892,2025-03-27,400.00,1,0,1,Direct Deposit,-1
3764,1490892,2025-03-31,1565.88,1,0,1,Direct Deposit,-1
3765,1490892,2025-03-31,321.10,0,0,1,Direct Deposit,-1


In [57]:
payments[payments["M_IDX"].isin(two_midx)].groupby("M_IDX")["AmountPaid"].sum()

M_IDX
1169872    3423.71
1490892    2286.98
Name: AmountPaid, dtype: float64

In [58]:
enriched_combined.columns[enriched_combined.columns.str.contains("Balance", case=False)]

Index(['ML1_MaxOpeningBalance', 'ML1_TotalOpeningBalances',
       'ML1_DebtRecoveryBalanceEscalation',
       'ML1_SingleCreditFacitlityBalanceEscalation',
       'ML1_GarageBalanceEscalation', 'ML1_LifeInsuranceBalanceEscalation',
       'ML1_OnemonthpersonalLoanBalanceEscalation',
       'ML1_SecuredPension_PolicyBackedLendingBalanceEscalation',
       'ML1_OpenLimitlessBalanceEscalation',
       'ML1_PersonalLoanBalanceEscalation',
       'ML1_StudentLoansBalanceEscalation', 'ML1_UtilityBalanceEscalation',
       'ML1_OverdraftBalanceEscalation', 'ML1_RentalsAssetBalanceEscalation',
       'ML1_RentalsPropertyBalanceEscalation',
       'ML1_RevolvingCreditNonStoreBalanceEscalation',
       'ML1_ShortTermInsuranceBalanceEscalation',
       'ML1_InstallmentBalanceEscalation',
       'ML1_VehicleFinanceBalanceEscalation',
       'ML1_OpenServicesBalanceEscalation', 'ML1_HomeLoanBalanceEscalation',
       'ML1_RevolvingCredit_StoreCardsBalanceEscalation',
       'ML1_CreditCardBalanceE

In [59]:
"M_IDX" in enriched_combined.columns

True

In [60]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(enriched_combined["M_IDX"]).sum()

np.int64(929)

**Finding:** After removing the 9,381 confirmed retracted accounts, 939 matters remained with a missing `CurrentBalance`. Of these, 937 also had a missing `FirstPaymentDate`, while 2 had a recorded `FirstPaymentDate` and corresponding payment records. The 937 matters were all `Closed`, had no matching payment or PTP records, and showed a similar date pattern. However, the available evidence was insufficient to classify them definitively as retracted accounts. They will therefore be retained at this stage and revisited after the datasets have been integrated and the eligibility criteria for the modelling population have been established.



#### 4.4 Negative Current Balances

Negative `CurrentBalance` values may indicate that a matter has been overpaid. Their relationship with the current status of the matter is investigated before deciding whether these records should be excluded or treated differently.


In [61]:
matters[matters["CurrentBalance"] < 0]["CurrentStatus"].value_counts()

CurrentStatus
Closed                 95
Broken PTP             13
Follow-up PTP          10
On Hold                 9
Attempting PTP          8
New Instruction         3
In Progress             2
Request for Closure     1
Name: count, dtype: int64

In [62]:
matters.loc[
    matters["CurrentBalance"] < 0,
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].head(20)

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
3,865730,Closed,1539.77,-300.00,2025-03-10
35,865933,Closed,4840.40,-0.40,2025-02-19
42,865982,Closed,1659.24,-0.76,2025-02-17
47,866003,Closed,1318.56,-564.38,2025-02-28
53,866028,Closed,402.98,-0.02,2025-03-18
302,824164,Closed,152.93,-7.07,2025-05-16
397,866167,Closed,152.41,-0.59,2025-04-07
402,866339,Closed,809.81,-0.19,2025-03-17
413,866372,Closed,352.32,-1788.89,2025-04-03
425,867065,Closed,656.32,-3.68,2025-03-27


In [63]:
matters["M_IDX"].isna().sum()

np.int64(0)

In [64]:
negative_balance = matters["CurrentBalance"] < 0
negative_balance.sum()

np.int64(141)

In [65]:
matters.loc[
    (matters["CurrentBalance"] < 0) &
    (matters["CurrentStatus"] != "Closed"),
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].sort_values("CurrentBalance")

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
9684,1169866,Broken PTP,2234.02,-20015.30,2025-02-27
53230,1055658,Follow-up PTP,12268.24,-8173.01,2025-04-16
10593,1238265,Broken PTP,1147.59,-1147.59,2025-04-30
5154,1200393,On Hold,4890.20,-1021.27,2025-03-25
34913,1656284,Broken PTP,313.99,-1000.00,2025-05-10
13126,1520039,Attempting PTP,542.20,-957.80,NaT
4543,1170155,Attempting PTP,1717.97,-943.01,2025-03-01
2406,1042753,Attempting PTP,8300.64,-921.83,2025-02-18
4203,1193861,Attempting PTP,13950.40,-851.92,NaT
53243,1169108,Follow-up PTP,1535.66,-831.39,2025-03-06


**Finding:** 141 matters had a negative `CurrentBalance`. Although 46 of these matters were not recorded as `Closed` in the source data, the negative balances indicate that the amount collected exceeded the remaining balance. Following the business rule used in the original project, these matters are treated as financially settled and their `CurrentStatus` is therefore set to `Closed` in the cleaned dataset.


In [66]:
matters.loc[matters["CurrentBalance"] < 0, "CurrentStatus"] = "Closed"

In [67]:
matters.loc[
    matters["CurrentBalance"] < 0,
    "CurrentStatus"
].value_counts()

CurrentStatus
Closed    141
Name: count, dtype: int64

In [68]:
matters["ActivationPeriod"].describe()

count    3771.000000
mean        0.720764
std         0.770642
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: ActivationPeriod, dtype: float64

In [69]:
matters["ActivationPeriod"].value_counts(dropna=False).sort_index()

ActivationPeriod
0.0     1699
1.0     1520
2.0      458
3.0       94
NaN    43027
Name: count, dtype: int64

In [70]:
matters.loc[
    matters["ActivationPeriod"].notna() &
    matters["FirstPaymentDate"].notna(),
    ["HandoverDate", "FirstPaymentDate", "ActivationPeriod"]
].head(10)

,HandoverDate,FirstPaymentDate,ActivationPeriod
0,2025-02-11,2025-02-27,0.0
1,2025-02-11,2025-02-19,0.0
2,2025-02-11,2025-02-20,0.0
3,2025-02-12,2025-03-10,1.0
4,2025-02-12,2025-02-28,0.0
5,2025-02-12,2025-02-24,0.0
6,2025-02-12,2025-02-25,0.0
7,2025-02-12,2025-02-28,0.0
8,2025-02-12,2025-03-14,1.0
9,2025-02-12,2025-03-11,1.0


In [71]:
calculated_activation = (
    (matters["FirstPaymentDate"].dt.year - matters["HandoverDate"].dt.year) * 12
    + (matters["FirstPaymentDate"].dt.month - matters["HandoverDate"].dt.month)
)

activation_check = (
    calculated_activation == matters["ActivationPeriod"]
)

activation_check[matters["ActivationPeriod"].notna()].value_counts()

True    3771
Name: count, dtype: int64

**Finding:** `ActivationPeriod` represents the number of calendar months between `HandoverDate` and `FirstPaymentDate`. All 3,771 non-missing `ActivationPeriod` values were validated against these two dates and matched exactly. Missing values were retained because they indicate that a `FirstPaymentDate` is not available, rather than representing an activation period of zero.


In [72]:
matters.loc[
    matters["FirstPaymentDate"].notna() &
    (matters["FirstPaymentDate"] < matters["HandoverDate"]),
    ["M_IDX", "HandoverDate", "FirstPaymentDate", "ActivationPeriod"]
]

,M_IDX,HandoverDate,FirstPaymentDate,ActivationPeriod


In [73]:
matters["CurrentStatus"].value_counts()

CurrentStatus
Attempting PTP         26466
Closed                  6547
New Instruction         6338
Payment Arrangement     3633
Broken PTP              2804
Re-opened                292
Authentication           271
In Progress              230
On Hold                  151
Follow-up PTP             60
Request for Closure        5
Dispute                    1
Name: count, dtype: int64

In [74]:
activation_can_be_calculated = (
    matters["ActivationPeriod"].isna()
    & matters["HandoverDate"].notna()
    & matters["FirstPaymentDate"].notna()
)

activation_can_be_calculated.sum()

np.int64(0)

**Finding:** `ActivationPeriod` represents the number of calendar months between `HandoverDate` and `FirstPaymentDate`. All 3,771 non-missing `ActivationPeriod` values were validated against these two dates and matched exactly. No records had a missing `ActivationPeriod` while both `HandoverDate` and `FirstPaymentDate` were available, so there were no additional values that could be reliably calculated. Missing values were therefore retained because the required date information was not available.


In [75]:
matters[["HandoverDate", "FirstPaymentDate", "DateCreated"]].agg(["min", "max"])

,HandoverDate,FirstPaymentDate,DateCreated
min,2025-02-03,2025-02-03,2025-05-23
max,2025-05-22,2025-05-22,2025-05-23


In [76]:
matters.loc[
    (matters["HandoverDate"] > matters["DateCreated"]) |
    (matters["FirstPaymentDate"] > matters["DateCreated"]),
    ["M_IDX", "HandoverDate", "FirstPaymentDate", "DateCreated"]
]

,M_IDX,HandoverDate,FirstPaymentDate,DateCreated


### 4.6 Matters Cleaning Completion

The `Matters` dataset has been reviewed for duplicate records, missing identifiers, balance inconsistencies, retracted accounts, negative balances, and date-related inconsistencies. Confirmed retracted accounts were removed, negative-balance matters were classified as `Closed`, and `ActivationPeriod` was validated against its underlying dates.

Records with missing `CurrentBalance` were retained for further consideration during dataset integration and definition of the modelling population. No additional date inconsistencies were identified.

The resulting dataset will be carried forward for integration with the remaining source datasets.


### 4.7 Matters Cleaning Completion

The `Matters` dataset has been cleaned and validated based on the identified data-quality and business-rule requirements. Duplicate matter identifiers were not found, confirmed retracted accounts were removed, negative `CurrentBalance` values were treated as financially settled matters and classified as `Closed`, and `ActivationPeriod` was validated against the underlying dates.

Records with missing `CurrentBalance` were retained because there was insufficient evidence to classify them as invalid at this stage. They will be revisited when the datasets are integrated and the modelling population is defined.

The cleaned `Matters` dataset will now be saved to the `data/processed/` directory. The original raw dataset in `data/raw/` remains unchanged so that the cleaning process can be reproduced.

In [108]:
matters.shape

(46798, 12)

#### Saving the Cleaned Dataset

The cleaned `Matters` dataset is saved as an Excel file in `data/processed/`. This creates a reproducible output of the cleaning process while preserving the original raw dataset.

In [110]:
matters.to_excel("../data/processed/Matters_cleaned.xlsx", index=False)

In [111]:
pd.read_excel("../data/processed/Matters_cleaned.xlsx").shape

(46798, 12)

#### Saved Dataset Verification

The saved `Matters` dataset is reloaded and compared with the cleaned DataFrame to confirm that the exported file preserves the cleaned data.

In [112]:
matters_saved = pd.read_excel("../data/processed/Matters_cleaned.xlsx")

matters.equals(matters_saved)

False

In [113]:
pd.DataFrame({
    "original": matters.dtypes,
    "saved": matters_saved.dtypes
})

,original,saved
M_IDX,int64,int64
HandoverDate,datetime64[us],datetime64[us]
CurrentStatusID,int64,int64
CurrentStatus,str,str
PreviousStatusID,float64,float64
PreviousStatus,str,str
FirstPaymentDate,datetime64[us],datetime64[us]
ActivationPeriod,float64,float64
OpeningBalance,float64,float64
CurrentBalance,float64,float64


In [114]:
import numpy as np

np.allclose(
    matters.select_dtypes(include="number"),
    matters_saved.select_dtypes(include="number"),
    equal_nan=True
)

True

**Finding:** The cleaned `Matters` dataset was successfully exported to `data/processed/Matters_cleaned.xlsx` and reloaded with the expected 46,798 rows and 12 columns. The data types were preserved and the numeric values were confirmed to match within floating-point tolerance. The processed file will be used as the cleaned output while the original raw dataset remains unchanged.

## 5. Payments Cleaning

The `Payments` dataset contains payment transactions associated with individual matters. The dataset will be reviewed for duplicate records, missing or inconsistent values, invalid dates or amounts, and other data-quality issues that could affect the analysis and subsequent integration with the other debt-collection datasets.

Cleaning decisions will be based on the structure of the data, business context, and the requirements of the intended prediction task. The original raw dataset will remain unchanged, while all cleaning will be performed on the working DataFrame.


### 5.1 Initial Data Quality Checks

Before applying any transformations, the structure and completeness of the `Payments` dataset will be assessed. This provides a baseline for identifying missing values, unexpected data types, and potential data-quality issues before cleaning.


In [77]:
payments.isna().sum()

M_IDX                  0
PaymentDate            0
AmountPaid             0
IsPayedAtClient        0
IsDebitOrderPayment    0
PaymentMethodID        0
PaymentDescription     0
AgentID                0
dtype: int64

In [78]:
payments.duplicated().sum()

np.int64(88)

In [79]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 6016 entries, 0 to 6015
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   M_IDX                6016 non-null   int64         
 1   PaymentDate          6016 non-null   datetime64[us]
 2   AmountPaid           6016 non-null   float64       
 3   IsPayedAtClient      6016 non-null   int64         
 4   IsDebitOrderPayment  6016 non-null   int64         
 5   PaymentMethodID      6016 non-null   int64         
 6   PaymentDescription   6016 non-null   str           
 7   AgentID              6016 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(5), str(1)
memory usage: 376.1 KB


In [80]:
payments[payments.duplicated(keep=False)]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
60,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
61,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
62,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
94,825013,2025-05-06,500.00,1,0,0,Unspecified,-1
95,825013,2025-05-06,500.00,1,0,0,Unspecified,-1
...,...,...,...,...,...,...,...,...
5594,1665470,2025-05-06,350.00,1,0,1,Direct Deposit,-1
5938,1793102,2025-05-17,154.05,1,0,0,Unspecified,-1
5939,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1
5940,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1


In [81]:
payments[payments.duplicated(keep=False)]["M_IDX"].value_counts()

M_IDX
868845     7
1660223    7
1663838    7
1205026    6
1660241    6
1660330    6
1660523    6
1660551    6
1662294    6
1665422    6
1665470    6
954855     4
1400821    4
1660489    4
1665152    4
1793102    4
824697     3
908344     3
1167111    3
825013     2
866080     2
908338     2
957569     2
972763     2
974298     2
974349     2
989785     2
1167200    2
1169873    2
1169951    2
1170155    2
1171057    2
1176684    2
1197580    2
1200689    2
1231503    2
1238299    2
1378705    2
1520037    2
1556599    2
1648059    2
1649784    2
1650468    2
1656338    2
1660340    2
1660528    2
1660665    2
1662207    2
1662328    2
Name: count, dtype: int64

In [82]:
payments[payments["M_IDX"] == 868845]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
440,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
441,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
442,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
443,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
444,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
445,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
446,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283


In [83]:
payments[payments["M_IDX"] == 868845].index

RangeIndex(start=440, stop=447, step=1)

In [84]:
payments.drop_duplicates().shape

(5928, 8)

### 5.2 Duplicate Payment Records

Exact duplicate rows were identified by comparing all fields in the `Payments` dataset. Because a matter can legitimately have multiple payment transactions, `M_IDX` was not used as a duplicate identifier.

A total of 88 exact duplicate rows were identified. After removing these duplicates, the dataset contains 5,928 rows across 8 columns.

These records are considered redundant because they contain identical information across all payment fields. Exact duplicate rows will therefore be removed from the working dataset.

In [85]:
payments = payments.drop_duplicates()

In [87]:
payments.shape

(5928, 8)

In [88]:
payments.duplicated().sum()

np.int64(0)

### 5.3 Duplicate Removal

The 88 exact duplicate payment records were removed from the working `Payments` dataset. The resulting dataset contains 5,928 rows and 8 columns. A subsequent duplicate check confirmed that no exact duplicate rows remain.

The original raw `Payments` dataset was not modified.


In [90]:
payments = payments.drop_duplicates()

In [91]:
payments.duplicated().sum()

np.int64(0)

### 5.4 Payment Amount Checks

The `AmountPaid` field represents the monetary value associated with each payment transaction. The distribution of payment amounts will be examined to identify negative values, zero values, unusually large payments, and other potentially anomalous values.

These values will be assessed in the context of the payment data and business rules before any records are modified or excluded.


In [92]:
payments["AmountPaid"].describe()

count     5928.000000
mean       699.138548
std       1993.526180
min     -12000.000000
25%        160.000000
50%        300.000000
75%        772.072500
max      60664.000000
Name: AmountPaid, dtype: float64

In [93]:
payments[payments["AmountPaid"] < 0]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
354,868079,2025-05-03,-300.00,1,0,0,Unspecified,-1
444,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
545,893887,2025-05-21,-326.03,0,0,1,Direct Deposit,74811
699,897402,2025-05-05,-500.00,0,1,29,DebiCheck Batched,73872
960,953986,2025-03-27,-1514.94,0,1,39,RMS,77547
...,...,...,...,...,...,...,...,...
5680,1667153,2025-05-06,-222.22,0,0,1,Direct Deposit,-1
5684,1667179,2025-05-06,-250.00,0,0,1,Direct Deposit,-1
5806,1713470,2025-05-10,-490.18,0,1,39,RMS,77818
5939,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1


In [96]:
negative_amounts = payments.loc[payments["AmountPaid"] < 0, "AmountPaid"].abs().unique()

negative_amounts

array([  300.  ,   581.85,   326.03,   500.  ,  1514.94,   704.  ,
         750.  ,   550.  ,  1000.  ,  1662.  ,   350.  ,   610.  ,
         600.  ,   700.  ,  2755.17,   430.  ,  1100.  ,   450.  ,
         400.  ,  1376.  ,   436.59,   200.  ,   800.  ,   978.  ,
         150.  ,  1500.  ,  2000.  ,   940.  , 12000.  ,  1388.  ,
        2500.  ,  2726.  ,  3122.74,   620.  ,   100.  ,  1300.  ,
         250.  ,    82.09,    38.88,    26.21, 10205.55,  2774.39,
         745.77,   125.  ,  1352.36,  7123.05,  1090.29,  1184.18,
         915.85,  1430.54,  4287.77,   705.68,  1231.75,   320.  ,
         667.51,  2040.96,  1200.  ,   918.06,  2163.62,   591.19,
         532.99,   277.28,   671.96,  3577.96,  1414.15,  2323.54,
        5052.07,   760.95,   205.08,  2359.  ,  2357.4 ,  3467.48,
        1530.41,  1635.61,  1516.1 ,  3142.71,   406.46,  1628.35,
        1019.53,  2186.06,   369.  ,  1135.43,   195.  ,   915.23,
        2698.28,  2147.19,   425.64,  1623.33,  7144.24,  1049

In [97]:
negative_payments = payments[payments["AmountPaid"] < 0].copy()

negative_payments["matching_positive"] = negative_payments.apply(
    lambda row: (
        (payments["M_IDX"] == row["M_IDX"]) &
        (payments["AmountPaid"] == abs(row["AmountPaid"]))
    ).any(),
    axis=1
)

negative_payments["matching_positive"].value_counts()

matching_positive
True     449
False      3
Name: count, dtype: int64

In [98]:
negative_payments[negative_payments["matching_positive"] == False]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID,matching_positive
2673,1388320,2025-03-11,-82.09,0,0,0,Unspecified,-1,False
2754,1390367,2025-03-11,-38.88,0,0,0,Unspecified,76714,False
3001,1402386,2025-02-22,-26.21,0,0,0,Unspecified,77249,False


In [99]:
payments[
    payments["M_IDX"].isin([1388320, 1390367, 1402386])
].sort_values(["M_IDX", "PaymentDate"])

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
2673,1388320,2025-03-11,-82.09,0,0,0,Unspecified,-1
2674,1388320,2025-03-11,20944.97,0,0,1,Direct Deposit,-1
2753,1390367,2025-03-11,2870.00,0,0,1,Direct Deposit,76714
2754,1390367,2025-03-11,-38.88,0,0,0,Unspecified,76714
3000,1402386,2025-02-22,1470.00,0,0,1,Direct Deposit,77249
3001,1402386,2025-02-22,-26.21,0,0,0,Unspecified,77249


**Finding:** Negative payment amounts were investigated rather than automatically treated as invalid records. Of the 452 negative payment transactions, 449 had an exact matching positive payment for the same matter and amount, indicating that they are likely associated with payment reversals or adjustments. The remaining 3 negative transactions did not have an exact amount match, but each occurred on the same date as a positive payment for the same matter and was associated with the same agent. These records were therefore retained, as there was insufficient evidence to classify them as erroneous. Removing negative transactions could also distort the actual amount collected.

### 5.5 Zero Payment Amounts

A payment transaction with an `AmountPaid` of zero does not represent a monetary payment. These records will therefore be investigated to determine whether they are legitimate transaction records, administrative entries, or potential data-quality issues before deciding whether they should be removed.

In [100]:
payments[payments["AmountPaid"] == 0]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID


**Finding:** No payment transactions had an `AmountPaid` value of zero. No records were removed based on zero payment amounts.

### 5.6 Unusually Large Payment Amounts

The `AmountPaid` distribution contains a small number of relatively large transactions. These records will be investigated to determine whether they represent legitimate payments or potential data-quality issues. Large payment amounts will not be removed solely because they are unusual, as debt amounts can vary substantially between matters.

In [101]:
payments.nlargest(10, "AmountPaid")

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
4124,1520319,2025-05-12,60664.00,0,0,1,Direct Deposit,76713
4287,1556524,2025-04-27,55635.30,1,0,0,Unspecified,-1
153,862317,2025-02-15,39054.80,1,0,0,Unspecified,-1
4754,1660209,2025-05-07,33445.06,0,0,1,Direct Deposit,75892
64,824701,2025-03-07,25791.70,1,0,0,Unspecified,-1
4410,1627745,2025-04-26,24575.30,1,0,0,Unspecified,-1
1648,1169866,2025-04-08,22535.00,1,0,1,Direct Deposit,-1
1259,1002772,2025-02-27,21306.60,1,0,1,Direct Deposit,-1
2674,1388320,2025-03-11,20944.97,0,0,1,Direct Deposit,-1
4345,1566778,2025-04-25,17865.50,0,0,1,Direct Deposit,77554


In [102]:
large_payments = payments.nlargest(10, "AmountPaid")

large_payments.merge(
    matters[["M_IDX", "OpeningBalance", "CurrentBalance", "CurrentStatus"]],
    on="M_IDX",
    how="left"
)[[
    "M_IDX",
    "AmountPaid",
    "OpeningBalance",
    "CurrentBalance",
    "CurrentStatus"
]]

,M_IDX,AmountPaid,OpeningBalance,CurrentBalance,CurrentStatus
0,1520319,60664.00,74978.80,14438.10,Closed
1,1556524,55635.30,1245.89,245.89,Attempting PTP
2,862317,39054.80,2503.67,3.67,Closed
3,1660209,33445.06,32886.16,48.35,Broken PTP
4,824701,25791.70,6335.03,-25475.16,Closed
5,1627745,24575.30,1884.07,-17000.00,Closed
6,1169866,22535.00,2234.02,-20015.30,Closed
7,1002772,21306.60,44255.14,21323.89,Payment Arrangement
8,1388320,20944.97,25394.54,5017.22,Closed
9,1566778,17865.50,21542.62,0.00,Closed


**Finding:** The largest payment transactions were reviewed against the corresponding matter balances. Several large payments exceeded the original `OpeningBalance` and resulted in negative `CurrentBalance` values, while others were within the original balance. These values are therefore consistent with legitimate payment activity, including overpayments, rather than indicating obvious data-entry errors. No payment records were removed based on payment size.

### 5.7 Payment Date Checks

Payment dates are reviewed to identify transactions that fall outside the expected data-collection period. The overall date range will first be examined, followed by checks against the corresponding matter's `HandoverDate` and the project pull date.

In [103]:
payments["PaymentDate"].agg(["min", "max"])

min   2025-02-06
max   2025-05-22
Name: PaymentDate, dtype: datetime64[us]

In [104]:
payment_handover_check = payments.merge(
    matters[["M_IDX", "HandoverDate"]],
    on="M_IDX",
    how="left"
)

payment_handover_check.loc[
    payment_handover_check["PaymentDate"] < payment_handover_check["HandoverDate"],
    ["M_IDX", "PaymentDate", "HandoverDate"]
]

,M_IDX,PaymentDate,HandoverDate


**Finding:** No payment transactions occurred before the corresponding matter's `HandoverDate`. The payment dates are therefore temporally consistent with the matter handover dates.

In [105]:
payments.loc[
    payments["PaymentDate"] > pd.Timestamp("2025-05-23"),
    ["M_IDX", "PaymentDate", "AmountPaid"]
]

,M_IDX,PaymentDate,AmountPaid


**Finding:** No payment transactions occurred after the project pull date of 23 May 2025. The payment data therefore falls within the expected observation period.

In [106]:
payments["M_IDX"].nunique(), len(payments)

(3786, 5928)

In [107]:
payments.loc[
    ~payments["M_IDX"].isin(matters["M_IDX"]),
    "M_IDX"
].nunique()

0

**Finding:** All payment transactions have a corresponding `M_IDX` in the cleaned `Matters` dataset. No unmatched payment records were identified, confirming that the `Payments` dataset can be linked to `Matters` using `M_IDX`.

### 5.8 Payments Cleaning Completion

The `Payments` dataset has been reviewed for missing values, duplicate records, payment amount anomalies, and temporal inconsistencies. Exact duplicate rows were removed, while negative and unusually large payment amounts were investigated and retained because they were consistent with legitimate payment activity.

No zero-value payments were identified. Payment dates were within the expected project period, no payments occurred before the corresponding matter's `HandoverDate`, and no payments occurred after the 23 May 2025 project pull date. All payment records also had a corresponding `M_IDX` in the cleaned `Matters` dataset.

The cleaned `Payments` dataset will now be saved to the `data/processed/` directory. The original raw dataset remains unchanged.

#### Final Dataset Check

Before saving the cleaned dataset, the final number of rows and columns is checked to confirm that the expected cleaning steps have been applied.

In [117]:
payments.shape

(5928, 8)

#### Saving the Cleaned Dataset

The cleaned `Payments` dataset is saved as an Excel file in `data/processed/`. This creates a reproducible output of the cleaning process while preserving the original raw dataset.

In [118]:
payments.to_excel("../data/processed/Payments_cleaned.xlsx", index=False)

In [119]:
payments_saved = pd.read_excel("../data/processed/Payments_cleaned.xlsx")

payments_saved.shape

(5928, 8)

#### Saved Dataset Verification

The saved `Payments` dataset is reloaded and its structure is compared with the cleaned DataFrame to confirm that the exported file preserves the expected data types and values.

In [120]:
pd.DataFrame({
    "original": payments.dtypes,
    "saved": payments_saved.dtypes
})

,original,saved
M_IDX,int64,int64
PaymentDate,datetime64[us],datetime64[us]
AmountPaid,float64,float64
IsPayedAtClient,int64,int64
IsDebitOrderPayment,int64,int64
PaymentMethodID,int64,int64
PaymentDescription,str,str
AgentID,int64,int64


In [121]:
np.allclose(
    payments.select_dtypes(include="number"),
    payments_saved.select_dtypes(include="number"),
    equal_nan=True
)

True

**Finding:** The cleaned `Payments` dataset was successfully exported to `data/processed/Payments_cleaned.xlsx` and reloaded with the expected 5,928 rows and 8 columns. The data types were preserved and the numeric values were confirmed to match within floating-point tolerance. The processed file will be used as the cleaned output while the original raw dataset remains unchanged.